## Pivoting data frames 
### Error of the Week 5 - Python

Let's look at some pivoting errors in Python!

The data we are using we created in Error-of-the-Week05.RMD. You may wish to complete the R Error of the Week first to better understand the data. Here will be pivoting wider the data that we made long in the R notebook. We have the observation variables `Question` and `Answer` as well as ID variables `id`, `age`, and `sex`. 

Let's say I have some data representing a questionnaire gathered from 20 participants. 5 questions were asked alongside demographic information of age and sex. 

In [1]:
import pandas as pd

In [3]:
# read in data, found in the data folder 
qdata = pd.read_csv("../data/data_long.csv")

In [4]:
# now lets do some data checks to see what we are working with 
qdata.head(20)

,id,age,sex,Question,Answer
0,1,31,M,Q_1,1
1,1,31,M,Q_2,12
2,1,31,M,Q_3,1
3,1,31,M,Q_4,9
4,1,31,M,Q_5,12
5,2,45,M,Q_1,4
6,2,45,M,Q_2,7
7,2,45,M,Q_3,5
8,2,45,M,Q_4,9
9,2,45,M,Q_5,12


In [5]:
print(qdata.describe())

               id         age      Answer
count  100.000000  100.000000  100.000000
mean    10.500000   47.200000    6.130000
std      5.795331   14.219065    3.656059
min      1.000000   20.000000    1.000000
25%      5.750000   35.750000    3.000000
50%     10.500000   47.000000    5.500000
75%     15.250000   55.250000    9.000000
max     20.000000   77.000000   15.000000


In [6]:
print(qdata.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        100 non-null    int64 
 1   age       100 non-null    int64 
 2   sex       100 non-null    object
 3   Question  100 non-null    object
 4   Answer    100 non-null    int64 
dtypes: int64(3), object(2)
memory usage: 4.0+ KB
None


Let's just make sure everything is as expected - so let's create some categories

In [7]:
qdata.id = qdata.id.astype('category')
qdata.sex = qdata.sex.astype('category')

Check the data frame again

In [8]:
print(qdata.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   id        100 non-null    category
 1   age       100 non-null    int64   
 2   sex       100 non-null    category
 3   Question  100 non-null    object  
 4   Answer    100 non-null    int64   
dtypes: category(2), int64(2), object(1)
memory usage: 3.5+ KB
None


In [9]:
qdata.id

0      1
1      1
2      1
3      1
4      1
      ..
95    20
96    20
97    20
98    20
99    20
Name: id, Length: 100, dtype: category
Categories (20, int64): [1, 2, 3, 4, ..., 17, 18, 19, 20]

## Now let's pivot wider

In [10]:
qdata_wide = qdata.pivot(index = ["id", "age", "sex"], # remember we can pass a list to the index parameter 
                       columns = "Question",
                       values = "Answer")

Let's see if it worked!

In [11]:
print(qdata_wide)

Question    Q_1  Q_2  Q_3  Q_4  Q_5
id age sex                         
1  31  M      1   12    1    9   12
2  45  M      4    7    5    9   12
3  65  M      2    1    3    8   11
4  63  F      9    1    4    5    8
5  36  F      8   10    4    5    6
6  36  F      6    3    2    7    9
7  69  F      5    9    5    7    3
8  62  F      1    4    1   11    5
9  47  F      9   13    2    2    1
10 39  M     10    8    3    7    6
11 34  F     10   10    5   12   11
12 50  M      4   15    2    3   10
13 50  M      8    1    2    6   11
14 47  M      8    9    4    1   11
15 35  M      4    9    2    1    5
16 53  M      2    1    3    4   10
17 20  M      9   14    7    5    5
18 35  F     10    8    3   10    6
19 77  M      9    7    4    3    1
20 50  F     12    3    3    9    5


In [12]:
qdata_wide.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 20 entries, (1, 31, 'M') to (20, 50, 'F')
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Q_1     20 non-null     int64
 1   Q_2     20 non-null     int64
 2   Q_3     20 non-null     int64
 3   Q_4     20 non-null     int64
 4   Q_5     20 non-null     int64
dtypes: int64(5)
memory usage: 2.5 KB


In [13]:
qdata_wide.id

AttributeError: 'DataFrame' object has no attribute 'id'

Wait, is that not right? Why do we only have 5 columns as stated in the `.info` method, and why can we not access the `id` column? 

In [14]:
qdata_wide.Q_1

id  age  sex
1   31   M       1
2   45   M       4
3   65   M       2
4   63   F       9
5   36   F       8
6   36   F       6
7   69   F       5
8   62   F       1
9   47   F       9
10  39   M      10
11  34   F      10
12  50   M       4
13  50   M       8
14  47   M       8
15  35   M       4
16  53   M       2
17  20   M       9
18  35   F      10
19  77   M       9
20  50   F      12
Name: Q_1, dtype: int64

Okay, but I really want to be able to look at `id`, which clearly does not exist anymore... It lives in the index and as mentioned in the `.info` method, we now have a MultiIndex... so what can we do?

Let's try the `pd.DataFrame.reset_index`

In [15]:
help(pd.DataFrame.reset_index)

Help on function reset_index in module pandas.core.frame:

reset_index(self, level: 'IndexLabel | None' = None, *, drop: 'bool' = False, inplace: 'bool' = False, col_level: 'Hashable' = 0, col_fill: 'Hashable' = '', allow_duplicates: 'bool | lib.NoDefault' = <no_default>, names: 'Hashable | Sequence[Hashable] | None' = None) -> 'DataFrame | None'
    Reset the index, or a level of it.

    Reset the index of the DataFrame, and use the default one instead.
    If the DataFrame has a MultiIndex, this method can remove one or more
    levels.

    Parameters
    ----------
    level : int, str, tuple, or list, default None
        Only remove the given levels from the index. Removes all levels by
        default.
    drop : bool, default False
        Do not try to insert index into dataframe columns. This resets
        the index to the default integer index.
    inplace : bool, default False
        Whether to modify the DataFrame rather than creating a new one.
    col_level : int or s

In [16]:
qdata_wide_correct = qdata_wide.reset_index()

print(qdata_wide_correct)

Question  id  age sex  Q_1  Q_2  Q_3  Q_4  Q_5
0          1   31   M    1   12    1    9   12
1          2   45   M    4    7    5    9   12
2          3   65   M    2    1    3    8   11
3          4   63   F    9    1    4    5    8
4          5   36   F    8   10    4    5    6
5          6   36   F    6    3    2    7    9
6          7   69   F    5    9    5    7    3
7          8   62   F    1    4    1   11    5
8          9   47   F    9   13    2    2    1
9         10   39   M   10    8    3    7    6
10        11   34   F   10   10    5   12   11
11        12   50   M    4   15    2    3   10
12        13   50   M    8    1    2    6   11
13        14   47   M    8    9    4    1   11
14        15   35   M    4    9    2    1    5
15        16   53   M    2    1    3    4   10
16        17   20   M    9   14    7    5    5
17        18   35   F   10    8    3   10    6
18        19   77   M    9    7    4    3    1
19        20   50   F   12    3    3    9    5


Looks good! Let's check the info method of our dataframe

In [17]:
qdata_wide_correct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   id      20 non-null     category
 1   age     20 non-null     int64   
 2   sex     20 non-null     category
 3   Q_1     20 non-null     int64   
 4   Q_2     20 non-null     int64   
 5   Q_3     20 non-null     int64   
 6   Q_4     20 non-null     int64   
 7   Q_5     20 non-null     int64   
dtypes: category(2), int64(6)
memory usage: 1.9 KB


Let's see that we can get `id`

In [18]:
qdata_wide_correct.id

0      1
1      2
2      3
3      4
4      5
5      6
6      7
7      8
8      9
9     10
10    11
11    12
12    13
13    14
14    15
15    16
16    17
17    18
18    19
19    20
Name: id, dtype: category
Categories (20, int64): [1, 2, 3, 4, ..., 17, 18, 19, 20]

## Woo! Success! 

Always remember to check your index after pivoting data frames in `pandas`, whether you wish to have a MultiIndex or not! **Oftentimes times the most dangerous errors are those which do not result in an error message explicitly.**

---
### That's all for now! 

*Dr. Brittany Blankinship (2026)*